# Taller 5 — Evolución Completa de Agentes en el Dilema del Prisionero Iterado (IPD)
**De la Miopía Frecuentista (`Rulito`) y 1-Step Lookahead (`Cloud`), pasando por Búsqueda `Expectimax + Markov TFT`, hasta el Modelado del Entorno $P(O_t \mid s_{t-1}, A_t)$ y Aprendizaje por Refuerzo Online con Factor de Olvido**

Este notebook reúne de forma reproducible y autocontenida **todo el código de los 5 agentes desarrollados a lo largo del semestre**, las simulaciones del Torneo de Axelrod (1980), las pruebas en condiciones estacionarias y no estacionarias, las métricas de convergencia y la generación de todos los gráficos del informe:
1. **`RulitoMiope` ($\gamma = 0$):** Modelo ingenuo $P(O_t \mid s_{t-1})$ que maximiza recompensa inmediata y colapsa al equilibrio de Nash $(D, D)$.
2. **`CloudOneStep` ($\gamma = 0.90$, 1-step lookahead):** Incorpora visión a 1 paso sobre $P(O_t \mid s_{t-1})$, pero sufre por la causalidad incompleta y la exploración forzada ($t < 10$).
3. **`ExpectimaxMarkovAgent` ($H = 4$, $\gamma = 0.90$):** Árbol Expectimax de 256 hojas con matriz de *Tit For Tat* hardcodeada y cadena de Markov cerrada $(I_4 - \gamma P)^{-1} P R$ como heurística en las hojas.
4. **`EmpiricalPolicyAgent` (Agente Base Taller 5, $\pi_0 = \text{CDDC}$):** Cambio de paradigma al modelar la transición del entorno $P(O_t \mid s_{t-1}, A_t)$ ($8$ contextos $\to$ matriz $8 \times 4$) mediante conteo supervisado exacto ($N = 173{,}130$) y *Value Iteration* ($\gamma = 0.90$).
5. **`ForgettingTransitionRLAgent` (Agente RL Taller 5, $n_0 = 2.0, \lambda = 0.90$):** Actualización incremental *online* de la matriz $8 \times 4$ mediante contador efectivo con factor de olvido ($n_k = \lambda n_{k-1} + 1$, $\eta_k = 1/n_k$) y replanificación con *Value Iteration* turno a turno.

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import axelrod as axl

from ipd_mdp import (
    ACTION_C,
    ACTION_D,
    ACTION_LABELS,
    ACTIONS,
    C,
    D,
    PAYOFF,
    STATE_MAP,
    STATES,
    UNIFORM_PRIOR_P,
    build_mdp_transition_matrix,
    build_training_pool,
    collect_frequency_table,
    compute_cumulative_avg_scores,
    solve_bellman,
)
from agents import EmpiricalPolicyAgent, ForgettingTransitionRLAgent
from plot_learning import CooperateThenDefect, ThreePhaseRival, rolling_round_scores

print(f"Entorno cargado correctamente. Estados: {STATES} | Pagos: {PAYOFF}")

Entorno cargado correctamente. Estados: ('CC', 'CD', 'DC', 'DD') | Pagos: {(1, 1): 3.0, (1, 0): 0.0, (0, 1): 5.0, (0, 0): 1.0}


## 1. Primeras Generaciones: `Rulito` ($\gamma=0$), `Cloud` (1-Step Lookahead) y `Expectimax + Markov TFT`

### Deducción matemática del colapso de `Rulito` al Equilibrio de Nash
Cuando el agente estima únicamente $\hat{p}_s = \hat{P}(O_t = C \mid s_{t-1} = s)$ y maximiza el pago inmediato ($\gamma = 0$):
$$\hat{Q}(s, C) = 3\,\hat{p}_s + 0\,(1 - \hat{p}_s) = 3\,\hat{p}_s, \qquad \hat{Q}(s, D) = 5\,\hat{p}_s + 1\,(1 - \hat{p}_s) = 4\,\hat{p}_s + 1$$
Restando ambas expresiones:
$$\hat{Q}(s, D) - \hat{Q}(s, C) = (4\,\hat{p}_s + 1) - 3\,\hat{p}_s = \hat{p}_s + 1 \ge 1 > 0 \qquad \forall\, \hat{p}_s \in [0, 1]$$
Por tanto, sin importar lo que aprenda el modelo frecuentista, $D$ domina estrictamente a $C$ en todo estado visitado, haciendo que `Rulito` caiga inevitablemente en el equilibrio de Nash miope $(D, D)$.

In [2]:
class RulitoMiope(axl.Player):
    """Generacion 1: Agente frecuentista miope (gamma = 0) sobre P(O_t | s_{t-1})."""

    name = "1. Rulito (Miope g=0)"
    exploration_turns = 10
    update_interval = 5

    def __init__(self):
        super().__init__()
        self.reset_learning()

    def reset_learning(self):
        self.transition_counts = {}
        self.transition_model = {}
        self.last_state = None

    def reset(self):
        super().reset()
        self.reset_learning()

    def strategy(self, opponent):
        turn = len(self.history)
        if self.last_state is not None and opponent.history:
            resp = opponent.history[-1]
            cnt = self.transition_counts.setdefault(self.last_state, {C: 0, D: 0})
            cnt[resp] += 1
        if turn and turn % self.update_interval == 0:
            self.transition_model = {}
            for s, cnt in self.transition_counts.items():
                tot = cnt[C] + cnt[D]
                if tot > 0:
                    self.transition_model[s] = {C: cnt[C] / tot, D: cnt[D] / tot}
        state = (self.history[-1], opponent.history[-1]) if (self.history and opponent.history) else None
        self.last_state = state
        if turn < self.exploration_turns:
            return C if turn % 2 == 0 else D
        probs = self.transition_model.get(state)
        if probs is None:
            return C
        q_c = 3.0 * probs[C]
        q_d = 5.0 * probs[C] + 1.0 * probs[D]
        return D if q_d > q_c else C


class CloudOneStep(axl.Player):
    """Generacion 2: Agente con 1-Step Lookahead (gamma = 0.90) sobre P(O_t | s_{t-1})."""

    name = "2. Cloud (1-Step g=0.9)"
    exploration_turns = 10
    update_interval = 5
    gamma = 0.90

    def __init__(self):
        super().__init__()
        self.reset_learning()

    def reset_learning(self):
        self.transition_counts = {}
        self.transition_model = {}
        self.last_state = None

    def reset(self):
        super().reset()
        self.reset_learning()

    def _probs(self, state):
        return self.transition_model.get(state, {C: 0.5, D: 0.5})

    def _imm(self, act, probs):
        return 3.0 * probs[C] if act == C else (5.0 * probs[C] + 1.0 * probs[D])

    def _val(self, act, state):
        p = self._probs(state)
        imm = self._imm(act, p)
        fut = sum(
            p[r] * max(self._imm(C, self._probs((act, r))), self._imm(D, self._probs((act, r))))
            for r in (C, D)
        )
        return imm + self.gamma * fut

    def strategy(self, opponent):
        turn = len(self.history)
        if self.last_state is not None and opponent.history:
            resp = opponent.history[-1]
            cnt = self.transition_counts.setdefault(self.last_state, {C: 0, D: 0})
            cnt[resp] += 1
        if turn and turn % self.update_interval == 0:
            self.transition_model = {}
            for s, cnt in self.transition_counts.items():
                tot = cnt[C] + cnt[D]
                if tot > 0:
                    self.transition_model[s] = {C: cnt[C] / tot, D: cnt[D] / tot}
        state = (self.history[-1], opponent.history[-1]) if (self.history and opponent.history) else None
        self.last_state = state
        if turn < self.exploration_turns:
            return C if turn % 2 == 0 else D
        return D if self._val(D, state) > self._val(C, state) else C


STATES_ORDER = [(C, C), (C, D), (D, C), (D, D)]
REWARD_VEC = np.array([3.0, 0.0, 5.0, 1.0], dtype=np.float64)


class ExpectimaxMarkovAgent(axl.Player):
    """Generacion 3: Expectimax (H=4) + Heuristica de Cadena de Markov con matriz TFT hardcodeada."""

    name = "3. Expectimax H=4 + Markov"

    def __init__(self, horizon: int = 4, gamma: float = 0.90):
        super().__init__()
        self.horizon = horizon
        self.gamma = gamma
        self.reset_model()

    def reset_model(self):
        self.counts = {s: {C: 0, D: 0} for s in STATES_ORDER}
        self.last_state = None

    def reset(self):
        super().reset()
        self.reset_model()

    def _get_E(self):
        return {
            s: (self.counts[s][C] / (self.counts[s][C] + self.counts[s][D]))
            if (self.counts[s][C] + self.counts[s][D]) > 0
            else 0.0
            for s in STATES_ORDER
        }

    def _leaves(self, E):
        p_cc, p_cd, p_dc, p_dd = E[(C, C)], E[(C, D)], E[(D, C)], E[(D, D)]
        if p_dc > 0.70 and p_dd > 0.60:
            P = np.array(
                [[0, 0, p_cc, 1 - p_cc], [0, 0, p_cd, 1 - p_cd], [0, 0, p_dc, 1 - p_dc], [0, 0, p_dd, 1 - p_dd]],
                dtype=float,
            )
        else:
            P = np.array(
                [[p_cc, 1 - p_cc, 0, 0], [0, 0, p_cd, 1 - p_cd], [p_dc, 1 - p_dc, 0, 0], [0, 0, p_dd, 1 - p_dd]],
                dtype=float,
            )
        v = np.linalg.solve(np.eye(4) - self.gamma * P, P @ REWARD_VEC)
        gH = self.gamma ** self.horizon
        return {STATES_ORDER[i]: float(gH * v[i]) for i in range(4)}

    def _tree(self, d, s, g, disc, E, lh):
        if d == self.horizon:
            return (1.0 - self.gamma) * (g + lh[s])
        pc, pd = E[s], 1.0 - E[s]
        nd = disc * self.gamma
        vc = pc * self._tree(d + 1, (C, C), g + disc * 3.0, nd, E, lh) + pd * self._tree(d + 1, (C, D), g, nd, E, lh)
        vd = pc * self._tree(d + 1, (D, C), g + disc * 5.0, nd, E, lh) + pd * self._tree(
            d + 1, (D, D), g + disc * 1.0, nd, E, lh
        )
        return vc if vc >= vd else vd

    def strategy(self, opponent):
        if not self.history:
            self.reset_model()
            return C
        obs = (self.history[-1], opponent.history[-1])
        resp = opponent.history[-1]
        self.counts[self.last_state if self.last_state is not None else (C, C)][resp] += 1
        self.last_state = obs
        E = self._get_E()
        lh = self._leaves(E)
        pc, pd = E[self.last_state], 1.0 - E[self.last_state]
        fc = pc * self._tree(1, (C, C), 3.0, self.gamma, E, lh) + pd * self._tree(1, (C, D), 0.0, self.gamma, E, lh)
        fd = pc * self._tree(1, (D, C), 5.0, self.gamma, E, lh) + pd * self._tree(1, (D, D), 1.0, self.gamma, E, lh)
        return C if fc >= fd - 1e-6 else D


print("Clases historicas definidas: RulitoMiope, CloudOneStep, ExpectimaxMarkovAgent")

Clases historicas definidas: RulitoMiope, CloudOneStep, ExpectimaxMarkovAgent


## 2. Cambio de Paradigma: Modelar el Entorno $P(O_t \mid s_{t-1}, A_t)$ y Agente Base (`CDDC`)

En lugar de modelar $P(O_t \mid s_{t-1})$ desde la vista ingenua del agente, modelamos la transición real del entorno en la ventana $[t-1, t]$:
$$P(s_t = (A_t, O_t) \mid s_{t-1}, A_t) = P(O_t \mid s_{t-1}, A_t)$$
Sobre las $N = 173{,}130$ transiciones del pool de 30 estrategias, extraemos la matriz $8 \times 4$ inicial $T_0$ y aplicamos *Value Iteration* ($\gamma = 0.90$), de donde emerge sin hardcodear *Tit For Tat* la política base **Pavlov (`CDDC`)**.

In [3]:
pool = build_training_pool()
freq = collect_frequency_table(pool, turns=200, repetitions=1, seed=42)
bellman0 = solve_bellman(freq.p_coop, gamma=0.90)
pol_str = "".join(ACTION_LABELS[a] for a in bellman0.policy)

print(f"Transiciones recolectadas: {freq.total_transitions:,}")
print(f"Politica Base emergente pi_0 (CC, CD, DC, DD): {pol_str}")
print("V*(s):", [f"{v:.2f}" for v in bellman0.v_values])
print("Delta Q*(s) = Q*(s,C) - Q*(s,D):", [f"{dq:+.2f}" for dq in bellman0.delta_q])

Transiciones recolectadas: 173,130
Politica Base emergente pi_0 (CC, CD, DC, DD): CDDC
V*(s): ['27.50', '24.92', '24.77', '23.81']
Delta Q*(s) = Q*(s,C) - Q*(s,D): ['+1.07', '-0.56', '-0.56', '+0.61']


## 3. Evaluación Experimental Completa: Las 5 Generaciones en Axelrod 1980 y Duelos 1v1

In [4]:
agents = [
    ("1. Rulito (Miope g=0)", RulitoMiope()),
    ("2. Cloud (1-Step g=0.9)", CloudOneStep()),
    ("3. Expectimax H=4 + Markov", ExpectimaxMarkovAgent(horizon=4, gamma=0.90)),
    ("4. Base MDP Fija (CDDC)", EmpiricalPolicyAgent(policy=bellman0.policy, player_name="4. Base MDP Fija (CDDC)")),
    (
        "5. RL Olvido (n0=2, lam=0.9)",
        ForgettingTransitionRLAgent(
            prior_p=freq.p_coop,
            gamma=0.90,
            initial_count=2.0,
            forgetting_factor=0.90,
            player_name="5. RL Olvido (n0=2, lam=0.9)",
        ),
    ),
]

print("=== TORNEO OFICIAL DE AXELROD 1980 (16 jugadores, 200 turnos, 5 reps, seed=42) ===")
base_15 = [cls() for cls in axl.axelrod_first_strategies]
axl_summary = {}
for label, ag in agents:
    players = [p.clone() for p in base_15] + [ag.clone()]
    res = axl.Tournament(players=players, turns=200, repetitions=5, seed=42).play(progress_bar=False)
    rows = [
        (p.name, float(np.mean(res.normalised_scores[i])), float(np.std(res.normalised_scores[i])))
        for i, p in enumerate(players)
    ]
    rows.sort(key=lambda r: r[1], reverse=True)
    rank = next(idx + 1 for idx, r in enumerate(rows) if r[0] == ag.name)
    val = next(r for r in rows if r[0] == ag.name)
    axl_summary[label] = (rank, val[1], val[2])
    print(f"{label:<30} -> Puesto #{rank:2d}/16 | {val[1]:.3f} +- {val[2]:.3f} pts/ronda")

print("\n=== CONDICIONES DE PRUEBA 1v1 (200 turnos, Pts/Ronda) ===")
conds = [
    ("TitForTat", axl.TitForTat()),
    ("Grudger", axl.Grudger()),
    ("Defector", axl.Defector()),
    ("Alternator", axl.Alternator()),
    ("Cambio C->D", CooperateThenDefect(50)),
    ("3 Fases C-D-C", ThreePhaseRival()),
]
cond_scores = {label: {} for label, _ in agents}
for label, ag in agents:
    for cname, opp in conds:
        a_c, o_c = ag.clone(), opp.clone()
        axl.Match((a_c, o_c), turns=200, seed=42).play()
        cond_scores[label][cname] = round(compute_cumulative_avg_scores(a_c.history, o_c.history)[-1], 3)
    print(f"{label:<30} | {cond_scores[label]}")

=== TORNEO OFICIAL DE AXELROD 1980 (16 jugadores, 200 turnos, 5 reps, seed=42) ===
1. Rulito (Miope g=0)          -> Puesto #11/16 | 1.948 +- 0.014 pts/ronda
2. Cloud (1-Step g=0.9)        -> Puesto # 7/16 | 2.359 +- 0.019 pts/ronda
3. Expectimax H=4 + Markov     -> Puesto # 1/16 | 2.659 +- 0.072 pts/ronda
4. Base MDP Fija (CDDC)        -> Puesto # 3/16 | 2.559 +- 0.018 pts/ronda
5. RL Olvido (n0=2, lam=0.9)   -> Puesto # 1/16 | 2.662 +- 0.024 pts/ronda

=== CONDICIONES DE PRUEBA 1v1 (200 turnos, Pts/Ronda) ===
1. Rulito (Miope g=0)          | {'TitForTat': 1.12, 'Grudger': 1.01, 'Defector': 0.975, 'Alternator': 2.935, 'Cambio C->D': 1.93, '3 Fases C-D-C': 3.53}
2. Cloud (1-Step g=0.9)        | {'TitForTat': 2.975, 'Grudger': 1.01, 'Defector': 0.975, 'Alternator': 2.95, 'Cambio C->D': 1.935, '3 Fases C-D-C': 3.535}
3. Expectimax H=4 + Markov     | {'TitForTat': 3.0, 'Grudger': 3.0, 'Defector': 0.995, 'Alternator': 2.49, 'Cambio C->D': 1.495, '3 Fases C-D-C': 2.995}
4. Base MDP Fija (CD

## 4. Generación de Gráficos Comparativos y de Aprendizaje

Ejecutamos los generadores de las 3 figuras principales del informe (`fig_evolucion_agentes.png`, `resultados_experimento.png`, `aprendizaje_olvido.png` y `aprendizaje_detalle_rl.png`).

In [5]:
import run_experiment
import plot_learning

# 1. Generar fig_evolucion_agentes.png (comparativa de las 5 generaciones)
fig, axes = plt.subplots(2, 2, figsize=(14.5, 9.8))
fig.suptitle("Evolucion Metodologica de los Agentes en el Dilema del Prisionero Iterado", fontsize=13.5, fontweight="bold")
palette = ["#d32f2f", "#ef6c00", "#6a1b9a", "#0277bd", "#2e7d32"]

ax = axes[0, 0]
short_names = [
    "1. Rulito\n(Miope g=0)",
    "2. Cloud\n(1-Step g=0.9)",
    "3. Expectimax\nH=4 + Markov",
    "4. Base MDP\nFija (CDDC)",
    "5. RL Olvido\n(n0=2, lam=0.9)",
]
means = [axl_summary[l][1] for l, _ in agents]
stds = [axl_summary[l][2] for l, _ in agents]
ranks = [axl_summary[l][0] for l, _ in agents]
bars = ax.bar(range(len(agents)), means, yerr=stds, capsize=4, color=palette, alpha=0.9)
ax.axhline(2.560, color="black", linestyle="--", linewidth=1.1, label="Tit For Tat (2.560, #3/16)")
for b, m, s, r in zip(bars, means, stds, ranks):
    y_pos = m + s + 0.04 if abs(m - 2.359) > 0.01 else m - 0.34
    txt_col = "white" if abs(m - 2.359) <= 0.01 else "black"
    ax.text(
        b.get_x() + b.get_width() / 2,
        y_pos,
        f"{m:.3f}\n(#{r}/16)",
        ha="center",
        va="bottom",
        fontsize=8.2,
        fontweight="bold",
        color=txt_col,
    )
ax.set_xticks(range(len(agents)))
ax.set_xticklabels(short_names, fontsize=8)
ax.set_ylim(0.0, 3.12)
ax.set_ylabel("Puntos / Ronda (Axelrod 1980)")
ax.set_title("1. Progreso en el Torneo de Axelrod 1980 (16 jugadores, 5 reps)")
ax.legend(fontsize=8, loc="lower right")
ax.grid(axis="y", linestyle="--", alpha=0.4)

ax = axes[0, 1]
t200 = np.arange(1, 201)
for (label, ag), col in zip(agents, palette):
    a_c, o_c = ag.clone(), axl.Grudger()
    axl.Match((a_c, o_c), turns=200, seed=42).play()
    cum_pts = np.cumsum(
        [PAYOFF[(ACTION_C if x == C else ACTION_D, ACTION_C if y == C else ACTION_D)] for x, y in zip(a_c.history, o_c.history)]
    )
    ls = "--" if "Base" in label else "-"
    ax.plot(t200, cum_pts, label=f"{label} ({int(cum_pts[-1])} pts)", color=col, linestyle=ls, linewidth=2)
ax.axvline(10, color="gray", linestyle=":", linewidth=1.1, label="Fin exploracion Rulito/Cloud (t=10)")
ax.set_xlabel("Turno (t)")
ax.set_ylabel("Puntos Acumulados")
ax.set_title("2. Duelo vs Grudger: colapso por exploracion forzada vs cooperacion")
ax.legend(fontsize=7.6, loc="upper left")
ax.grid(alpha=0.3)

ax = axes[1, 0]
for (label, ag), col in zip(agents, palette):
    a_c, o_c = ag.clone(), axl.Defector()
    axl.Match((a_c, o_c), turns=200, seed=42).play()
    avgs = compute_cumulative_avg_scores(a_c.history, o_c.history)
    ls = "--" if "Base" in label else "-"
    ax.plot(t200, avgs, label=f"{label} ({avgs[-1]:.3f})", color=col, linestyle=ls, linewidth=2)
ax.set_xlabel("Turno (t)")
ax.set_ylabel("Puntos acumulados / ronda")
ax.set_title("3. Adaptacion vs Defector: vulnerabilidad de CDDC fija (0.500) vs RL")
ax.set_ylim(0.2, 1.08)
ax.legend(fontsize=7.6, loc="lower right")
ax.grid(alpha=0.3)

ax = axes[1, 1]
c_labels = ["TitForTat", "Grudger", "Defector", "Alternator", "Cambio C->D"]
x = np.arange(len(c_labels))
w = 0.155
for idx, ((label, _), col) in enumerate(zip(agents, palette)):
    vals = [cond_scores[label][c] for c in c_labels]
    ax.bar(x + (idx - 2) * w, vals, w, label=label, color=col, alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(c_labels, fontsize=8.2)
ax.set_ylabel("Puntos / Ronda (200 turnos)")
ax.set_title("4. Comparativa por Condicion Experimental (1v1, 200 turnos)")
ax.set_ylim(0.0, 3.72)
ax.legend(fontsize=7.3, loc="upper center", ncol=2, framealpha=0.95)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
fig.savefig("fig_evolucion_agentes.png", dpi=150)
plt.close(fig)

# 2. Generar curvas de aprendizaje online y mecanismo interno RL
plot_learning.main()
print("Figuras actualizadas exitosamente:\n - fig_evolucion_agentes.png\n - resultados_experimento.png\n - aprendizaje_olvido.png\n - aprendizaje_detalle_rl.png")

Figuras actualizadas exitosamente:
 - fig_evolucion_agentes.png
 - resultados_experimento.png
 - aprendizaje_olvido.png
 - aprendizaje_detalle_rl.png


## 5. Torneos con 10 Semillas Independientes (`seeds 0..9`), Benchmark de Harper et al. (2017) y Mega-Torneo de 175 Estrategias

Esta sección ejecuta:
1. **Primer Torneo de Axelrod (1980)** para **cada una de las 5 generaciones** promediando sobre **10 realizaciones independientes con 10 semillas distintas (`seeds = 0..9`)** (`fig_torneos_10seeds_5agentes.png`).
2. **Benchmark del Paper de Harper et al. (2017, arXiv:1707.06307)** con 41 estrategias (incluyendo agentes entrenados con RL/`EvolvedLookerUp2_2_2`, `EvolvedANN5`, `EvolvedFSM16`, `PSOGambler2_2_2`, `DBS`, `OmegaTFT`, `ZD-GTFT-2`, `Adaptive Pavlov`, `Gradual`) sobre 10 semillas.
3. **Mega-Torneo General de 175 Estrategias** (`170` estrategias de la librería `axelrod` + nuestras 5 generaciones), donde el aprendizaje *online* con factor de olvido impulsa al agente MDP **+77 puestos** (del `#100/175` al **`#23/175`**) y `Expectimax H=4 + Markov` alcanza el **`#10/175`** (`fig_megatorneo_harper.png`).

In [5]:
from run_tournaments_10seeds_and_harper import (
    SEEDS_10, run_tournament_10_seeds,
    plot_axelrod_10seeds_per_agent, plot_harper_and_mega_tournament
)
import json

with open('tournaments_10seeds_and_harper.json') as f:
    t_data = json.load(f)

print('=== 1. PRIMER TORNEO DE AXELROD (1980) POR AGENTE (10 SEMILLAS: SEEDS 0..9) ===')
for ag_name, ranking in t_data['individual_axelrod_10seeds'].items():
    r = next(x for x in ranking if x['name'] == ag_name)
    print(f"{ag_name:<33} -> Puesto #{r['overall_rank']:2d}/16 | Rank medio: {r['mean_rank']:.2f} | %#1: {r['win_rate_pct']:3.0f}% | Media 10 seeds: {r['mean_score']:.3f} ± {r['std_score']:.3f} pts/ronda")

print('\n=== 2. BENCHMARK DEL PAPER DE HARPER ET AL. (2017) (41 ESTRATEGIAS, 10 SEEDS) ===')
for r in t_data['harper_10seeds']:
    if r['overall_rank'] <= 15 or r['name'] in t_data['individual_axelrod_10seeds']:
        mark = ' <-- NUESTRO AGENTE' if r['name'] in t_data['individual_axelrod_10seeds'] else ''
        print(f"#{r['overall_rank']:2d}/41 {r['name']:<34} | {r['mean_score']:.3f} ± {r['std_score']:.3f} | Rank medio: {r['mean_rank']:.2f}{mark}")

print('\n=== 3. MEGA-TORNEO GENERAL DE 175 ESTRATEGIAS (LIBRERIA AXELROD + HARPER) ===')
for r in t_data['mega_tournament_175']:
    if r['overall_rank'] <= 15 or r['name'] in t_data['individual_axelrod_10seeds'] or r['name'] in ('First by Shubik', 'Forgiving Tit For Tat', 'Grudger'):
        mark = ' <-- NUESTRO AGENTE' if r['name'] in t_data['individual_axelrod_10seeds'] else ''
        print(f"#{r['overall_rank']:3d}/175 {r['name']:<34} | {r['mean_score']:.3f} pts/ronda{mark}")


=== 1. PRIMER TORNEO DE AXELROD (1980) POR AGENTE (10 SEMILLAS: SEEDS 0..9) ===
1. Rulito (Miope g=0)             -> Puesto #11/16 | Rank medio: 11.20 | %#1:   0% | Media 10 seeds: 1.921 ± 0.011 pts/ronda
2. Cloud (1-Step g=0.90)          -> Puesto # 6/16 | Rank medio: 6.10 | %#1:   0% | Media 10 seeds: 2.449 ± 0.032 pts/ronda
3. Expectimax H=4 + Markov        -> Puesto # 1/16 | Rank medio: 1.10 | %#1:  90% | Media 10 seeds: 2.686 ± 0.050 pts/ronda
4. Base MDP Fija (CDDC)           -> Puesto # 4/16 | Rank medio: 3.80 | %#1:  10% | Media 10 seeds: 2.539 ± 0.017 pts/ronda
5. RL Olvido (n0=2.0, lam=0.90)   -> Puesto # 1/16 | Rank medio: 1.30 | %#1:  70% | Media 10 seeds: 2.632 ± 0.014 pts/ronda

=== 2. BENCHMARK DEL PAPER DE HARPER ET AL. (2017) (41 ESTRATEGIAS, 10 SEEDS) ===
# 1/41 Omega TFT                          | 2.817 ± 0.011 | Rank medio: 1.20
# 2/41 3. Expectimax H=4 + Markov         | 2.808 ± 0.015 | Rank medio: 1.90 <-- NUESTRO AGENTE
# 3/41 EvolvedLookerUp2_2_2               |